# EAHN — Kaggle Training Notebook (Fixed Version)

**Run every cell top to bottom.**

This notebook uses the **fixed codebase** with the following improvements:
- ✅ Focal loss + label smoothing for class imbalance
- ✅ Frozen backbone (first 3 epochs) to prevent BatchNorm corruption
- ✅ Removed `cls_dropout` that caused train/test mismatch
- ✅ Gradient-alignment loss to couple attention with classification
- ✅ Class-conditional diversity loss to prevent class-agnostic collapse
- ✅ Stronger entropy penalty (`alpha=0.5`) to prevent one-hot collapse
- ✅ Weaker temporal consistency (`lambda2=0.05`) to allow frame variation

**Expected:** Epoch 1 Val AUC > 0.60. After 10 epochs AUC > 0.85.


In [ ]:
import os, glob

# ── Auto-detect FF++ data root ──────────────────────────────────────────────
def find_ffpp_root(base="/kaggle/input", max_depth=6):
    for root, dirs, _ in os.walk(base):
        depth = root.replace(base, "").count(os.sep)
        if depth > max_depth:
            dirs.clear()
            continue
        if (os.path.isdir(os.path.join(root, "original_sequences")) and
            os.path.isdir(os.path.join(root, "manipulated_sequences"))):
            return root
    return None

DATA_ROOT = find_ffpp_root()
REPO_URL = "https://github.com/umardrazbhatti/EAHNKimiCode.git"
REPO_DIR = "/kaggle/working/EAHNKimiCode"
OUTPUT_DIR = "/kaggle/working/outputs"
CACHE_DIR = "/kaggle/working/.face_cache"

# Use 2 for smoke-test; change to 10 for real results
EPOCHS = 2        # smoke test
BATCH_SIZE = 4

if DATA_ROOT is None:
    print("ERROR: FF++ root not found. Listing /kaggle/input for diagnosis:")
    for e in sorted(glob.glob("/kaggle/input/*/*")):
        print(f"  {e}")
    raise SystemExit("Set DATA_ROOT manually above after checking the listing.")

print("=" * 55)
print(f"DATA_ROOT : {DATA_ROOT}")
print(f"REPO_DIR  : {REPO_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"EPOCHS    : {EPOCHS}")
print(f"BATCH_SIZE: {BATCH_SIZE}")
print("=" * 55)
print("If DATA_ROOT looks wrong, set it manually and re-run only this cell.")


In [ ]:
import glob, os

METHODS = ["Deepfakes", "Face2Face", "FaceShifter", "FaceSwap", "NeuralTextures"]

real_dir = os.path.join(DATA_ROOT, "original_sequences", "youtube", "c23", "videos")
real_vids = glob.glob(os.path.join(real_dir, "*.mp4"))
status = "OK" if len(real_vids) > 0 else "MISSING"
print(f"[{status:^7}] {'original (real)':<22} {len(real_vids):>5} videos")

total_fake, all_ok = 0, len(real_vids) > 0
for method in METHODS:
    d = os.path.join(DATA_ROOT, "manipulated_sequences", method, "c23", "videos")
    count = len(glob.glob(os.path.join(d, "*.mp4")))
    total_fake += count
    status = "OK" if count > 0 else "MISSING"
    if count == 0: all_ok = False
    print(f"[{status:^7}] {method:<22} {count:>5} videos")

print(f"\nTotal real: {len(real_vids)} | Total fake: {total_fake} | Combined: {len(real_vids) + total_fake}")
if not all_ok:
    raise SystemExit("One or more directories missing. Fix DATA_ROOT in Cell 1.")
print("\nFile counts OK --- proceed to Cell 3.")


In [ ]:
import shutil, os

to_remove = [REPO_DIR, OUTPUT_DIR, CACHE_DIR, "/kaggle/working/eahn_outputs.zip"]
for path in to_remove:
    if os.path.isdir(path):
        shutil.rmtree(path)
        print(f"Removed dir : {path}")
    elif os.path.isfile(path):
        os.remove(path)
        print(f"Removed file: {path}")
    else:
        print(f"Not present : {path}")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
print("\nClean. Proceed to Cell 4.")


In [ ]:
import os, sys

ret = os.system(f"git clone {REPO_URL} {REPO_DIR}")
if ret != 0 or not os.path.isdir(REPO_DIR):
    raise SystemExit(
        "Clone failed.\n"
        "Fix: Enable Internet in Kaggle Settings (right panel → Internet → On).\n"
        "Then re-run this cell."
    )

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

print("\nClone successful. Repo contents:")
for f in sorted(os.listdir(REPO_DIR)):
    print(f"  {f}")


In [ ]:
import subprocess, sys, importlib

print("Installing requirements...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     f"{REPO_DIR}/requirements.txt"],
    check=True
)
print("Installation complete.\n")

pkg_map = {"torch":"torch","torchvision":"torchvision","timm":"timm",
           "sklearn":"sklearn","cv2":"cv2","PIL":"PIL","tqdm":"tqdm"}
for name, mod_name in pkg_map.items():
    try:
        m = importlib.import_module(mod_name)
        print(f" OK {name:<15} {getattr(m, '__version__', '?')}")
    except ImportError:
        print(f" MISSING {name}")

print("\nChecking full import chain...")
try:
    from config import EAHNConfig
    from data.datasets import DeepfakeDataset
    from data.collate import deepfake_collate_fn
    from data.transforms import get_transforms
    from data.face_align import FaceAligner
    from data.synthetic_generator import SyntheticDataGenerator
    from models.eahn import EAHN
    from losses.classification import build_classification_loss
    from losses.explanation import ExplanationLoss
    from losses.temporal import TemporalConsistencyLoss
    from metrics.detection import DetectionMetrics
    from scripts.train_real import main as train_main
    from scripts.evaluate import run_evaluation
    from scripts.dashboard import show_dashboard
    print("\nALL IMPORTS OK --- proceed to Cell 6.")
except ImportError as e:
    raise SystemExit(f"\nIMPORT FAILED: {e}")

config = EAHNConfig()
assert hasattr(config, 'attn_temp_init'), "attn_temp_init missing from EAHNConfig"
assert hasattr(config, 'attn_diversity_weight'), "attn_diversity_weight missing from EAHNConfig"
assert config.gamma == 0.1, f"gamma must be 0.1, got {config.gamma}"
print(f"Config verified: gamma={config.gamma}, temp_init={config.attn_temp_init}, diversity_w={config.attn_diversity_weight}")
print(f"Config: alpha={config.alpha}, lambda2={config.lambda2}, cls_loss_type={config.cls_loss_type}")
print(f"Config: freeze_backbone={config.freeze_backbone}, unfreeze_epoch={config.unfreeze_backbone_epoch}")


In [ ]:
import os, sys, torch

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

stale = [k for k in list(sys.modules) if any(k.startswith(p) for p in
         ["data","models","losses","xai","metrics","utils","scripts","config"])]
for k in stale: del sys.modules[k]

from config import EAHNConfig
from data.datasets import DeepfakeDataset
from data.collate import deepfake_collate_fn
from torch.utils.data import DataLoader

_cfg = EAHNConfig(
    data_root = DATA_ROOT,
    output_dir = OUTPUT_DIR,
    cache_dir = CACHE_DIR,
    dataset_name = "ff++",
    num_frames = 4,       # quick check only
    frame_size = 224,
    train_split = 0.8,
    val_split = 0.1,
    device = "auto",
    num_workers = 0,
)

print("Building dataset (train split)...")
ds = DeepfakeDataset(_cfg, mode="train", dataset_type="ff++")
loader = DataLoader(ds, batch_size=8, collate_fn=deepfake_collate_fn, shuffle=True)

print("Loading one batch...")
batch = next(iter(loader))
unique = batch["label"].unique().tolist()
print(f"\n Labels in batch : {unique}")
print(f" Frames shape    : {tuple(batch['frames'].shape)}")
print(f" Mask shape      : {tuple(batch['mask'].shape)}")
print(f" has_mask flags  : {batch['has_mask'].tolist()}")
print(f" Total samples   : {len(ds)}")

n_real = sum(1 for s in ds.samples if s["label"] == 0)
n_fake = sum(1 for s in ds.samples if s["label"] == 1)
print(f" Class split     : real={n_real} fake={n_fake}")

errors = []
if n_real == 0: errors.append("Real videos (label=0) missing from dataset split.")
if n_fake == 0: errors.append("Fake videos (label=1) missing from dataset split.")
if batch["frames"].shape[1] != 4: errors.append("Frame count mismatch.")
if batch["frames"].shape[2:] != torch.Size([3, 224, 224]):
    errors.append(f"Unexpected frame shape: {tuple(batch['frames'].shape[2:])}")

if errors:
    print("\n" + "="*55)
    print("VERIFICATION FAILED:")
    for e in errors: print(f" • {e}")
    print("="*55)
    raise SystemExit("Fix errors above before training.")

print("\n" + "="*55)
print("VERIFICATION PASSED --- both classes present, shapes correct.")
print("Proceed to Cell 7.")
print("="*55)


In [ ]:
import os, sys, shutil

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# Clear face cache to prevent T=4 (Cell 6) vs T=16 (training) collision
if os.path.isdir(CACHE_DIR):
    shutil.rmtree(CACHE_DIR)
os.makedirs(CACHE_DIR, exist_ok=True)
print("Face cache cleared.")

cmd = (
    f'python run_full_pipeline.py'
    f' --data_root "{DATA_ROOT}"'
    f' --dataset_name ff++'
    f' --output_dir "{OUTPUT_DIR}"'
    f' --cache_dir "{CACHE_DIR}"'
    f' --epochs {EPOCHS}'
    f' --batch_size {BATCH_SIZE}'
    f' --num_workers 0'
    f' --gamma 0.1'
    f' --attn_temp_init 1.386'
    f' --attn_diversity_weight 2.5'
    f' --alpha 0.5'              # FIXED: stronger entropy penalty
    f' --beta 0.5'
    f' --lambda2 0.05'           # FIXED: weaker temporal freeze
    f' --cls_loss_type focal'    # FIXED: focal loss for imbalance
    f' --focal_alpha 0.25'
    f' --focal_gamma 2.0'
    f' --freeze_backbone'        # FIXED: freeze backbone initially
    f' --unfreeze_backbone_epoch 3'
    f' --lambda_grad_align 0.1'  # FIXED: gradient alignment
    f' --label_smoothing 0.05'   # FIXED: label smoothing
    f' --eval_after_train'
)

print("Running:", cmd, "\n")
os.system(cmd)
print("\nCell 7 complete.")


In [ ]:
import pandas as pd, os

_csv = os.path.join(OUTPUT_DIR, "metrics.csv")
if not os.path.exists(_csv):
    print("metrics.csv not found --- did Cell 7 complete?")
else:
    _df = pd.read_csv(_csv, names=["metric","value"], header=0)
    def _get(m):
        row = _df[_df["metric"]==m]["value"]
        return float(row.values[0]) if len(row) else None

    _inter = _get("inter_sample_cosine_mean")
    _mode  = _get("peak_mode_share")
    _mt_std = _get("m_t_std_mean")
    _auc   = _get("auc_roc")
    _warnings = []

    if _inter is not None and _inter > 0.95:
        _warnings.append(f"inter_sample_cosine_mean={_inter:.3f}")
    if _mode is not None and _mode > 0.5:
        _warnings.append(f"peak_mode_share={_mode:.3f}")
    if _mt_std is not None and _mt_std > 0.13:
        _warnings.append(f"m_t_std_mean={_mt_std:.3f}")

    if _warnings:
        print("⚠️ EXPLANATION COLLAPSE DETECTED:")
        for w in _warnings: print(" -", w)
        print("Do NOT proceed to 10-epoch run. Diagnose the explanation head first.")
    else:
        print("✅ No collapse detected. Safe to proceed to longer runs.")

    if _auc is not None:
        print(f"\n📊 Detection AUC-ROC: {_auc:.4f}")
        if _auc >= 0.80: print("   → EXCELLENT")
        elif _auc >= 0.65: print("   → GOOD — run more epochs")
        elif _auc >= 0.55: print("   → IMPROVING — run more epochs")
        else: print("   → POOR — check data pipeline")


In [ ]:
import os
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

log_csv = os.path.join(OUTPUT_DIR, 'logs.csv')
if not os.path.exists(log_csv):
    print('No logs.csv found — training may not have started yet.')
else:
    df = pd.read_csv(log_csv, names=['step','tag','value'], header=0)
    
    # Parse tags like 'train/loss', 'val/auc_roc', 'train/lr'
    df['split'] = df['tag'].str.split('/').str[0]
    df['metric'] = df['tag'].str.split('/').str[1]
    
    # ── Plot 1: Loss curves ─────────────────────────────────────────────
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Training loss
    ax = axes[0, 0]
    train_loss = df[(df['split']=='train') & (df['metric']=='loss')]
    if len(train_loss):
        ax.plot(train_loss['step'], train_loss['value'], label='Train Loss', color='steelblue')
        ax.set_title('Training Loss per Epoch')
        ax.set_xlabel('Step')
        ax.set_ylabel('Loss')
        ax.legend()
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'No training loss data', ha='center', va='center', transform=ax.transAxes)
    
    # Validation AUC-ROC
    ax = axes[0, 1]
    val_auc = df[(df['split']=='val') & (df['metric']=='auc_roc')]
    if len(val_auc):
        ax.plot(val_auc['step'], val_auc['value'], marker='o', label='Val AUC-ROC', color='darkgreen')
        ax.axhline(0.5, color='red', linestyle='--', label='Chance')
        ax.axhline(0.8, color='green', linestyle='--', alpha=0.5, label='Good (>0.8)')
        ax.set_title('Validation AUC-ROC')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('AUC-ROC')
        ax.set_ylim(0.45, 1.0)
        ax.legend()
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'No validation AUC data', ha='center', va='center', transform=ax.transAxes)
    
    # Per-class accuracy
    ax = axes[1, 0]
    real_acc = df[(df['split']=='train') & (df['metric']=='real_acc')]
    fake_acc = df[(df['split']=='train') & (df['metric']=='fake_acc')]
    if len(real_acc) or len(fake_acc):
        if len(real_acc):
            ax.plot(real_acc['step'], real_acc['value'], marker='o', label='Real Acc', color='blue')
        if len(fake_acc):
            ax.plot(fake_acc['step'], fake_acc['value'], marker='s', label='Fake Acc', color='red')
        ax.set_title('Per-Class Training Accuracy')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Accuracy')
        ax.set_ylim(0, 1.05)
        ax.legend()
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'No per-class accuracy data', ha='center', va='center', transform=ax.transAxes)
    
    # Learning rate
    ax = axes[1, 1]
    lr_data = df[(df['split']=='train') & (df['metric']=='lr')]
    if len(lr_data):
        ax.plot(lr_data['step'], lr_data['value'], color='purple')
        ax.set_title('Learning Rate Schedule')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('LR')
        ax.set_yscale('log')
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'No LR data', ha='center', va='center', transform=ax.transAxes)
    
    plt.tight_layout()
    curves_path = os.path.join(OUTPUT_DIR, 'training_curves.png')
    plt.savefig(curves_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Training curves saved: {curves_path}')
    
    # Show inline
    from IPython.display import Image, display
    display(Image(filename=curves_path))


In [ ]:
import os, pandas as pd
from IPython.display import display

csv_path = os.path.join(OUTPUT_DIR, "metrics.csv")
if not os.path.exists(csv_path):
    print("metrics.csv not found. Did Cell 7 complete without errors?")
else:
    df = pd.read_csv(csv_path, header=None, names=["Metric", "Value"])
    df["Value"] = pd.to_numeric(df["Value"], errors="coerce").round(4)
    print("=" * 45)
    print(" EAHN Evaluation Metrics")
    print("=" * 45)
    display(df)

    auc_row = df.loc[df["Metric"] == "auc_roc", "Value"]
    if len(auc_row) and not pd.isna(auc_row.values[0]):
        v = auc_row.values[0]
        if v >= 0.80: msg = "EXCELLENT --- model is performing well."
        elif v >= 0.65: msg = "GOOD --- model is learning; more epochs will help."
        elif v >= 0.50: msg = "IMPROVING --- learning detected; run more epochs."
        else: msg = "POOR --- near chance; check data pipeline."
        print(f"\nAUC-ROC = {v:.4f} → {msg}")
    else:
        print("\nWARNING: AUC-ROC is NaN")


In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

graphs = [
    ("roc_curve.png", "ROC Curve"),
    ("pr_curve.png", "Precision-Recall Curve"),
    ("confusion_matrix.png", "Confusion Matrix"),
    ("confusion_matrix_norm.png", "Confusion Matrix (Normalised)"),
    ("score_distribution.png","Score Distribution"),
]

available = [(f, t) for f, t in graphs
             if os.path.exists(os.path.join(OUTPUT_DIR, f))]

if not available:
    print("No graph files found --- did Cell 7 produce output?")
else:
    n = len(available)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 5))
    if n == 1: axes = [axes]
    for ax, (fname, title) in zip(axes, available):
        ax.imshow(mpimg.imread(os.path.join(OUTPUT_DIR, fname)))
        ax.set_title(title, fontsize=12, fontweight="bold")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

    missing = [t for f, t in graphs if (f, t) not in available]
    if missing:
        print(f"Not yet generated: {missing}")


In [ ]:
import os
from IPython.display import Video, HTML, display

heatmap_dir = os.path.join(OUTPUT_DIR, "heatmaps")
if not os.path.isdir(heatmap_dir):
    print("No heatmaps directory found. Did Cell 7 complete?")
else:
    methods = ["intrinsic", "gradcam", "rollout", "shap"]
    mp4s = sorted(f for f in os.listdir(heatmap_dir) if f.endswith(".mp4"))
    vid_ids = sorted(set(
        f.replace(f"_{m}.mp4", "")
        for f in mp4s for m in methods
        if f.endswith(f"_{m}.mp4")
    ))

    if not vid_ids:
        print("No heatmap .mp4 files found in", heatmap_dir)
    else:
        sample = vid_ids[0]
        display(HTML(f"<h3>Sample: {sample} &nbsp;|&nbsp; {len(vid_ids)} total sample(s)</h3>"))
        for method in methods:
            path = os.path.join(heatmap_dir, f"{sample}_{method}.mp4")
            if os.path.exists(path):
                display(HTML(f"<p><b>{method.upper()}</b></p>"))
                display(Video(path, embed=True, width=480))
            else:
                print(f" [{method}] not found")


In [ ]:
import os, shutil

zip_out = "/kaggle/working/eahn_outputs.zip"
if os.path.exists(zip_out):
    os.remove(zip_out)

shutil.make_archive("/kaggle/working/eahn_outputs", 'zip', OUTPUT_DIR)
print(f"Zipped outputs: {zip_out}")
print(f"Size: {os.path.getsize(zip_out) / (1024*1024):.1f} MB")
print("\nRight panel → Output → Download 'eahn_outputs.zip'")
